# Cultural Idiom Feature Extraction Pipeline

This notebook implements the steps to extract deep cultural features using LLMs, multi-prompting, and embedding filtering.

## 1. Install Dependencies

In [3]:
!pip install google-generativeai groq sentence-transformers numpy scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.7/139.7 kB 3.2 MB/s eta 0:00:00


In [12]:
!pip install -U google-generativeai

## 2. Set Up Language Models and Embeddings

We will initialize the OpenAI API using your provided key and load the `LaBSE` model, which is highly recommended for multilingual embedding tasks (especially for languages like Hindi).

In [8]:
import os
from groq import Groq
from sentence_transformers import SentenceTransformer
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Setup your API Key here! (REDACTED FOR GITHUB)
GROQ_API_KEY = "YOUR_GROQ_API_KEY_HERE"

groq_client = Groq(api_key=GROQ_API_KEY)

print("Loading LaBSE embedding model (takes a minute)...")
embedding_model = SentenceTransformer('sentence-transformers/LaBSE')
print("Model loaded successfully!")

Loading LaBSE embedding model (takes a minute)...


model.safetensors:   0%|          | 0.00/1.88G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/LaBSE
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/397 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/114 [00:00<?, ?B/s]

2_Dense/model.safetensors:   0%|          | 0.00/2.36M [00:00<?, ?B/s]

Model loaded successfully!


## 3. Generate Multiple Variations (Multi-Prompting)

In [15]:
def get_cultural_features(idiom, tgt_lang="Hindi"):
    print(f"Extracting cultural features for: '{idiom}' using 3 different open-source models via Groq...\n")

    # Exact prompt from your base paper
    prompt = f"""You are a linguistic expert with deep knowledge of {tgt_lang} idioms, including their cultural and socio-cultural contexts. For the given idiom, provide a detailed analysis covering the following key aspects. Ensure each point has only a brief, single-sentence description:
    1. **Idiom:** - {idiom}
    2. **Concepts:** Explain the basic meaning and underlying concepts of the idiom.
    3. **Values:** Describe the beliefs or desirable outcomes that the idiom reflects.
    4. **Situational Context:** Describe typical scenarios where the idiom is used.
    5. **Historical Context:** Provide any relevant historical background influencing the idiom's usage."""

    glosses = []

    # Model 1: Llama 3.3 70B (Groq)
    try:
        print("1. Fetching from Llama 3.3 70B...")
        response_llama3 = groq_client.chat.completions.create(
            messages=[{"role": "user", "content": prompt}],
            model="llama-3.3-70b-versatile",
        )
        glosses.append(response_llama3.choices[0].message.content.strip())
        print("✅ Llama 3.3 70B done!")
    except Exception as e:
        print(f"❌ Llama 3.3 error: {e}")

    # Model 2: Kimi K2 (Groq)
    try:
        print("2. Fetching from Kimi K2...")
        response_kimi = groq_client.chat.completions.create(
            messages=[{"role": "user", "content": prompt}],
            model="moonshotai/kimi-k2-instruct-0905",
        )
        glosses.append(response_kimi.choices[0].message.content.strip())
        print("✅ Kimi K2 done!")
    except Exception as e:
        print(f"❌ Kimi K2 error: {e}")

    # Model 3: Llama 4 Scout (Groq)
    try:
        print("3. Fetching from Llama 4 Scout...")
        response_llama4 = groq_client.chat.completions.create(
            messages=[{"role": "user", "content": prompt}],
            model="meta-llama/llama-4-scout-17b-16e-instruct",
        )
        glosses.append(response_llama4.choices[0].message.content.strip())
        print("✅ Llama 4 Scout done!")
    except Exception as e:
        print(f"❌ Llama 4 Scout error: {e}")

    print("\nSuccessfully extracted features from all 3 models!")
    return glosses

# Test it with your idiom!
idiom_to_test = "राज़ खोलना"
candidate_glosses = get_cultural_features(idiom_to_test)

# Print the outputs to verify before embedding
for i, text in enumerate(candidate_glosses):
    print(f"\n--- Model {i+1} Output ---")
    print(text)

Extracting cultural features for: 'राज़ खोलना' using 3 different open-source models via Groq...

1. Fetching from Llama 3.3 70B...
✅ Llama 3.3 70B done!
2. Fetching from Kimi K2...
✅ Kimi K2 done!
3. Fetching from Llama 4 Scout...
✅ Llama 4 Scout done!

Successfully extracted features from all 3 models!

--- Model 1 Output ---
Here's the analysis of the idiom "राज़ खोलना" (Raj Khulna):
1. **Idiom:** The idiom "राज़ खोलना" translates to "to reveal a secret" in English.
2. **Concepts:** The idiom conceptually represents the act of uncovering or disclosing hidden information or a secret that was previously unknown or concealed.
3. **Values:** The idiom reflects the values of honesty, transparency, and trust, as revealing secrets is often associated with building or maintaining relationships.
4. **Situational Context:** The idiom is typically used in situations where someone is finally sharing confidential information, confessing to a mistake, or exposing a hidden truth.
5. **Historical Co

## 4. Convert to Embeddings, Filter (Denoise), and Ensemble

Check agreement -> Drop Outliers -> Ensemble the Winners.

In [16]:
def process_embeddings_and_filter(glosses, similarity_threshold=0.7):
    """Converts glosses to embeddings, removes outliers, and averages the remaining ones."""
    if not glosses:
        return None

    print("\nCalculating embeddings using LaBSE...")
    embeddings = embedding_model.encode(glosses)

    # Calculate cosine similarity matrix
    similarity_matrix = cosine_similarity(embeddings)

    print("\nSimilarity Matrix:")
    print(np.round(similarity_matrix, 2))

    n = len(glosses)
    if n <= 1:
        return embeddings[0]

    valid_indices = []
    for i in range(n):
        avg_sim = (np.sum(similarity_matrix[i]) - 1) / (n - 1)
        if avg_sim >= similarity_threshold:
            valid_indices.append(i)
            print(f"Keeping Model {i+1}'s feature (Avg Similarity: {avg_sim:.2f})")
        else:
            print(f"Dropping Model {i+1}'s feature as outlier (Avg Similarity: {avg_sim:.2f})")

    if not valid_indices:
        print("Warning: All variations were dropped! Returning average of all for fallback.")
        valid_indices = list(range(n))

    # Ensemble the Winners (Averaging)
    valid_embeddings = embeddings[valid_indices]
    final_embedding = np.mean(valid_embeddings, axis=0)

    print(f"\nFinal ensembled embedding shape: {final_embedding.shape}")
    return final_embedding

# Run the pipeline!
final_feature_vector = process_embeddings_and_filter(candidate_glosses)


Calculating embeddings using LaBSE...

Similarity Matrix:
[[1.   0.84 0.88]
 [0.84 1.   0.86]
 [0.88 0.86 1.  ]]
Keeping Model 1's feature (Avg Similarity: 0.86)
Keeping Model 2's feature (Avg Similarity: 0.85)
Keeping Model 3's feature (Avg Similarity: 0.87)

Final ensembled embedding shape: (768,)
